In [ ]:
!pip install -q -U \
    langchain \
    langchain-openai \
    langgraph \
    feedparser \
    beautifulsoup4 \
    requests \
    python-docx

In [ ]:
import os
from google.colab import userdata

# OpenAI
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# Gmail sender
os.environ["EMAIL_ADDRESS"] = userdata.get("EMAIL_ADDRESS")

# Gmail App Password
os.environ["EMAIL_APP_PASSWORD"] = userdata.get("EMAIL_APP_PASSWORD")

print("✅ Secrets loaded successfully!")

✅ Secrets loaded successfully!


In [ ]:
import os
import re
import smtplib
import requests
import feedparser

from datetime import datetime
from email.message import EmailMessage
from pathlib import Path

from bs4 import BeautifulSoup
from docx import Document
from docx.shared import Pt

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

from langgraph.graph import StateGraph, START, END

from typing import TypedDict, List

In [ ]:
llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0
)

print("AI model ready!")

AI model ready!


In [ ]:
@tool
def search_ai_news(topic: str = "Artificial Intelligence") -> str:
    """
    Search for recent AI news using Google News RSS.
    """

    url = "https://news.google.com/rss/search"

    params = {
        "q": f"{topic} when:1d",
        "hl": "en-US",
        "gl": "US",
        "ceid": "US:en"
    }

    response = requests.get(
        url,
        params=params,
        timeout=20
    )

    response.raise_for_status()

    feed = feedparser.parse(response.text)

    results = []

    for entry in feed.entries[:10]:

        title = entry.get("title", "")
        link = entry.get("link", "")
        published = entry.get("published", "")
        source = entry.get("source", {}).get(
            "title",
            "Unknown"
        )

        results.append(
            f"TITLE: {title}\n"
            f"SOURCE: {source}\n"
            f"DATE: {published}\n"
            f"URL: {link}"
        )

    return "\n\n".join(results)

In [ ]:
@tool
def fetch_article(url: str) -> str:
    """
    Fetch and extract readable text from a news article.
    """

    try:

        response = requests.get(
            url,
            timeout=20,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        for tag in soup([
            "script",
            "style",
            "noscript",
            "header",
            "footer",
            "nav"
        ]):
            tag.decompose()

        text = " ".join(
            soup.stripped_strings
        )

        return text[:12000]

    except Exception as e:

        return f"Unable to fetch article: {e}"

In [ ]:
@tool
def analyze_news(news_data: str) -> str:
    """
    Analyze collected AI news and identify the most important stories.
    """

    prompt = f"""
You are an AI news analyst.

Analyze the following recent AI news:

{news_data}

Your tasks:

1. Identify the most important stories.
2. Remove duplicate or very similar stories.
3. Categorize each story:
   - Generative AI
   - AI Research
   - AI Companies
   - AI Products
   - Robotics
   - Regulation
   - Business
   - Other

4. Rank stories by importance.

Return the result as a structured list.
"""

    response = llm.invoke([
        SystemMessage(
            content="You are an expert AI technology news analyst."
        ),
        HumanMessage(
            content=prompt
        )
    ])

    return response.content

In [ ]:
@tool
def summarize_news(news_analysis: str) -> str:
    """
    Create concise professional summaries of the selected AI news.
    """

    prompt = f"""
Create a professional AI news digest based on:

{news_analysis}

For every important story include:

### Headline
### Source
### What Happened
### Why It Matters

Keep each story concise.

At the end include:

## AI Trend of the Day

Explain the biggest overall trend
you see across today's AI news.

Do NOT invent information.
"""

    response = llm.invoke([
        SystemMessage(
            content="You are a professional technology news editor."
        ),
        HumanMessage(
            content=prompt
        )
    ])

    return response.content

In [ ]:
@tool
def create_news_document(news_digest: str) -> str:
    """
    Create a professional Word document containing the AI news digest.
    """

    doc = Document()

    # Title
    title = doc.add_heading(
        "AI NEWS DAILY",
        0
    )

    subtitle = doc.add_paragraph(
        datetime.now().strftime(
            "%B %d, %Y"
        )
    )

    subtitle.style = "Subtitle"

    doc.add_paragraph(
        "Daily digest of the latest Artificial Intelligence news."
    )

    doc.add_heading(
        "Today's AI News",
        level=1
    )

    # Add digest
    for line in news_digest.split("\n"):

        line = line.strip()

        if not line:
            continue

        if line.startswith("###"):
            doc.add_heading(
                line.replace("#", "").strip(),
                level=2
            )

        elif line.startswith("##"):
            doc.add_heading(
                line.replace("#", "").strip(),
                level=1
            )

        else:
            doc.add_paragraph(line)

    # Footer
    doc.add_paragraph()

    doc.add_paragraph(
        "Generated automatically by AI News Digest Agent."
    )

    path = "/content/AI_News_Digest.docx"

    doc.save(path)

    return path

In [ ]:
@tool
def send_email(
    subject: str,
    body: str,
    attachment_path: str
) -> str:
    """
    Send the generated AI news document by Gmail.
    """

    sender = os.environ["EMAIL_ADDRESS"]
    password = os.environ["EMAIL_APP_PASSWORD"]
    recipient = "bushratalaqalsulami@gmail.com"

    msg = EmailMessage()

    msg["Subject"] = subject
    msg["From"] = sender
    msg["To"] = recipient

    msg.set_content(body)

    with open(
        attachment_path,
        "rb"
    ) as file:

        file_data = file.read()

    msg.add_attachment(
        file_data,
        maintype="application",
        subtype="vnd.openxmlformats-officedocument.wordprocessingml.document",
        filename="AI_News_Digest.docx"
    )

    with smtplib.SMTP_SSL(
        "smtp.gmail.com",
        465
    ) as smtp:

        smtp.login(
            sender,
            password
        )

        smtp.send_message(msg)

    return "Email sent successfully!"

In [ ]:
class NewsState(TypedDict, total=False):

    topic: str

    raw_news: str

    analyzed_news: str

    final_digest: str

    document_path: str

    email_status: str

    trace: List[str]

In [ ]:
def news_research_agent(
    state: NewsState
):

    topic = state.get(
        "topic",
        "Artificial Intelligence"
    )

    print("🔎 Searching for latest AI news...")

    news = search_ai_news.invoke(
        topic
    )

    trace = state.get(
        "trace",
        []
    )

    trace.append(
        "News Research Agent: searched for recent AI news."
    )

    return {
        "raw_news": news,
        "trace": trace
    }

In [ ]:
def news_analysis_agent(
    state: NewsState
):

    print("🧠 Analyzing news...")

    analysis = analyze_news.invoke(
        state["raw_news"]
    )

    trace = state.get(
        "trace",
        []
    )

    trace.append(
        "News Analysis Agent: ranked and categorized news stories."
    )

    return {
        "analyzed_news": analysis,
        "trace": trace
    }

In [ ]:
def editor_agent(
    state: NewsState
):

    print("✍️ Creating AI news digest...")

    digest = summarize_news.invoke(
        state["analyzed_news"]
    )

    trace = state.get(
        "trace",
        []
    )

    trace.append(
        "Editor Agent: summarized and formatted the selected stories."
    )

    return {
        "final_digest": digest,
        "trace": trace
    }

In [ ]:
def document_agent(
    state: NewsState
):

    print("📄 Creating document...")

    path = create_news_document.invoke(
        state["final_digest"]
    )

    trace = state.get(
        "trace",
        []
    )

    trace.append(
        "Document Agent: generated the AI News Digest document."
    )

    return {
        "document_path": path,
        "trace": trace
    }

In [ ]:
def email_agent(
    state: NewsState
):

    print("📧 Sending email...")

    today = datetime.now().strftime(
        "%B %d, %Y"
    )

    body = f"""
Hello,

Here is your AI News Daily Digest for {today}.

The attached document contains:
- Latest AI news
- Important developments
- News analysis
- AI trends

This digest was generated automatically
by the AI News Digest Agent.

Best regards,
AI News Agent
"""

    status = send_email.invoke({

        "subject": f"🤖 AI News Daily — {today}",

        "body": body,

        "attachment_path":
            state["document_path"]
    })

    trace = state.get(
        "trace",
        []
    )

    trace.append(
        "Email Agent: sent the generated AI News Digest."
    )

    return {
        "email_status": status,
        "trace": trace
    }

In [ ]:
workflow = StateGraph(
    NewsState
)

workflow.add_node(
    "research",
    news_research_agent
)

workflow.add_node(
    "analysis",
    news_analysis_agent
)

workflow.add_node(
    "editor",
    editor_agent
)

workflow.add_node(
    "document",
    document_agent
)

workflow.add_node(
    "email",
    email_agent
)

workflow.add_edge(
    START,
    "research"
)

workflow.add_edge(
    "research",
    "analysis"
)

workflow.add_edge(
    "analysis",
    "editor"
)

workflow.add_edge(
    "editor",
    "document"
)

workflow.add_edge(
    "document",
    "email"
)

workflow.add_edge(
    "email",
    END
)

news_system = workflow.compile()

print("✅ AI News Agent workflow ready!")

✅ AI News Agent workflow ready!


In [ ]:
result = news_system.invoke({

    "topic": "Artificial Intelligence",

    "trace": []
})

print("\n" + "=" * 70)
print("AI NEWS DIGEST")
print("=" * 70)

print(result["final_digest"])

🔎 Searching for latest AI news...
🧠 Analyzing news...
✍️ Creating AI news digest...
📄 Creating document...
📧 Sending email...

AI NEWS DIGEST
1) 
### Headline
The turbulent AI era is here. The choices we make now are critical.
### Source
Gates Notes — 09 Sep 2026
### What Happened
Bill Gates published a policy‑oriented essay framing the current moment as a pivotal, turbulent era for AI and urging careful choices about governance and safety.
### Why It Matters
A high‑profile public voice like Gates can shape public debate, philanthropy, and policy direction during a critical decision window for AI governance and safety.

2)
### Headline
Yes, We’re Entering the Era of Artificial General Intelligence
### Source
Wall Street Journal — 09 Sep 2026
### What Happened
The WSJ published a major piece asserting that we are entering an AGI era, signaling a shift in mainstream media framing.
### Why It Matters
A mainstream outlet declaring an AGI‑era shift can accelerate investor, policy, and resea